# 🔬 GNOT RF Cavity Neural Operator — Colab Runner

**Branch:** `claude/spectral-subspace-architecture`  
**Mimari:** Frekans-Önce, Altuzay Decoder

**Ne değişti:**
- `mode_idx` girdi olarak kaldırıldı — model geometriden tüm 3 modu birden tahmin ediyor
- **Evrensel Grassmannian kaybı** — iyi-ayrılmış modda standart kayba indirgenir, dejenere modda altuzay mesafesi kullanılır
- Hangi modun dejenere olduğu hardcode edilmiyor; TARGET frekanslardan her geometri için ayrı belirleniyor
- `degeneracy_mode: soft | hard` config ile seçilir

**Loss Fonksiyonu:**
$$\mathcal{L}_{total} = \mathcal{L}_{Grassmannian} + \alpha \cdot \mathcal{L}_{freq} + \lambda \cdot \mathcal{L}_{bnd}$$


## 1. GPU Kontrolü & Ortam Kurulumu

In [ ]:
# GPU kontrol
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Repo klonla (veya Google Drive'dan bağla)
import os

REPO_URL = "https://github.com/KorayGokceler/rf_cavity_neural_operator.git"
BRANCH = "claude/spectral-subspace-architecture"
REPO_DIR = "/content/rf_cavity_neural_operator"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}
print(f"\nÇalışma dizini: {os.getcwd()}")
!git log --oneline -3


In [ ]:
# Bağımlılıkları kur
!pip install -q scikit-fem[all] gmsh pytorch-lightning torchmetrics seaborn tqdm pyyaml h5py

## 2. Veri Hazırlığı

Eğer verin yoksa:
1. **Veri üret** (dataset_generator.py → .h5)
2. **Features çıkar** (dataset_converter.py → .pkl)

Eğer verin varsa **(Google Drive'dan)**: 3. adıma atla.

In [ ]:
# === OPSIYON A: Veri Üretimi (sıfırdan) ===
# Bu ~15-30 dakika sürer (1000 geometri)

GENERATE_DATA = False  # True yap eğer sıfırdan veri üreteceksen

if GENERATE_DATA:
    # 1. Mesh + FEM çözümleri üret
    !python -c "
from src.data_gen.dataset_generator import generate_dataset
from src.config import load_config
cfg = load_config('configs/default.yaml')
generate_dataset(cfg.data_gen)
"
    # 2. GNOT formatına dönüştür — TÜM 3 MOD (0 1 2) dahil edilmeli
    !python convert.py \
        --h5_filepath rf_cavity_1000_dataset.h5 \
        --output_path data/gnot_dataset.pkl \
        --output_format pkl \
        --modes 0 1 2

    print("\n✅ Veri üretimi tamamlandı!")


In [ ]:
# === OPSIYON B: Google Drive'dan veri yükle ===

LOAD_FROM_DRIVE = True  # True yap eğer Drive'dan yükleyeceksen

if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Drive'daki veri yolunu güncelle:
    DRIVE_PKL = "/content/drive/MyDrive/rf_cavity_data/gnot_dataset.pkl"  # ← BUNU GÜNCELLE
    
    os.makedirs("data", exist_ok=True)
    if os.path.exists(DRIVE_PKL):
        !cp "{DRIVE_PKL}" data/gnot_dataset.pkl
        print(f"✅ Veri kopyalandı: {DRIVE_PKL} → data/gnot_dataset.pkl")
    else:
        print(f"❌ Dosya bulunamadı: {DRIVE_PKL}")
        print("Drive'daki doğru yolu gir!")

In [ ]:
# Veri kontrolü
import pickle
import numpy as np

DATA_PATH = "data/gnot_dataset.pkl"

with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

n_geoms = len(data['geometry_pool'])
n_samples = len(data['samples'])

# Feature boyutu
first_geom = list(data['geometry_pool'].values())[0]
n_features = first_geom['Input_funcs'].shape[1]
n_nodes_range = [g['X'].shape[0] for g in data['geometry_pool'].values()]

# Yeni formatta her sample tüm K modu içeriyor — Y_field [N, K], Y_freq [K]
first_sample = data['samples'][0]
K = first_sample['Y_field'].shape[-1]  # kaç mod

print(f"📊 Dataset Özeti:")
print(f"   Geometri sayısı:  {n_geoms}")
print(f"   Toplam sample:    {n_samples}  (1 sample = 1 geometri × {K} mod)")
print(f"   Feature boyutu:   {n_features}")
print(f"   Mod sayısı (K):   {K}")
print(f"   Node aralığı:     {min(n_nodes_range)} – {max(n_nodes_range)}")
print(f"   Y_field shape:    [{first_sample['Y_field'].shape}]  (N × K)")
print(f"   Y_freq shape:     [{first_sample['Y_freq'].shape}]  (K,)")

# Frekans dağılımı
freqs = np.array([s['Y_freq'] for s in data['samples']])  # [N_samples, K]
for k in range(K):
    print(f"   Mod {k} frekans (norm):  mean={freqs[:,k].mean():.3f}  std={freqs[:,k].std():.3f}")


## 3. Eğitim Konfigürasyonu

In [ ]:
# Konfigürasyon özeti
import yaml
with open('configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

m = cfg['model']
t = cfg['training']
print(f"Model     : shared={m['n_shared_layers']} layers, mode_branch={m['n_mode_layers']}, embed={m['embed_dim']}")
print(f"Modlar    : num_field_modes={m['num_field_modes']}")
print(f"Kayıp     : freq_w={t['freq_weight']},  degeneracy_mode={t['degeneracy_mode']}")
if t['degeneracy_mode'] == 'hard':
    print(f"            near_deg_threshold={t['near_deg_threshold']}")
else:
    print(f"            deg_sigma_rel={t['deg_sigma_rel']}")
print(f"Eğitim    : lr={t['learning_rate']},  batch={t['batch_size']},  epochs={t['max_epochs']}")


## 4. Hızlı Pipeline Testi

Tam eğitim başlamadan önce 1 iterasyon çalıştırarak her şeyin doğru olduğunu doğrula.

In [ ]:
# Fast dev run: 1 training + 1 validation iterasyonu
!python train.py --config configs/default.yaml --override training.num_workers=2 --fast_dev_run

print("\n✅ Pipeline testi başarılı! Tüm bileşenler çalışıyor.")

## 5. Tam Eğitim

In [ ]:
# Tam eğitim başlat
!python train.py --config configs/default.yaml --override training.num_workers=2

In [ ]:
# TensorBoard ile eğitim takibi (paralel çalıştır)
%load_ext tensorboard
%tensorboard --logdir training_logs

## 6. Inference & Görselleştirme

In [ ]:
# En iyi checkpoint'u bul
import glob, os

ckpt_pattern = "training_logs/gnot_5k_v1/**/best-*.ckpt"
ckpts = glob.glob(ckpt_pattern, recursive=True)

if ckpts:
    best_ckpt = sorted(ckpts)[-1]
    print(f"En iyi checkpoint: {best_ckpt}")
else:
    print("❌ Checkpoint bulunamadı. Eğitim tamamlandı mı?")
    best_ckpt = None


In [ ]:
# Inference çalıştır
if best_ckpt:
    !python infer.py \
        --checkpoint "{best_ckpt}" \
        --data_path data/gnot_dataset_5k.pkl \
        --split val \
        --num_samples 5 \
        --output_dir inference_results
else:
    print("Önce eğitimi çalıştır.")


In [ ]:
# Sonuçları göster
from IPython.display import Image, display
import glob

result_files = sorted(glob.glob("inference_results/*.png"))[:5]

for f in result_files:
    print(f"\n{'='*60}")
    print(f"📄 {os.path.basename(f)}")
    print(f"{'='*60}")
    display(Image(filename=f, width=800))

## 7. Altuzay & Frekans Değerlendirmesi

Model tarafından tahmin edilen modların kalitesini değerlendir:
- **Frekans hatası**: tahmin vs gerçek frekanslar (Hungarian eşleme sonrası)
- **Alan hatası (rel-L2)**: eşleştirilmiş modlar arası
- **Near-degenerate pairs**: yakın frekanslı mod çiftlerinde altuzay tutarlılığı


In [ ]:
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
from src.data.dataset import GNOTDataset, gnot_collate_fn
from src.training.lightning_module import GNOTLightning
from torch.utils.data import DataLoader

if best_ckpt:
    model = GNOTLightning.load_from_checkpoint(best_ckpt)
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)

    dataset = GNOTDataset("data/gnot_dataset.pkl", split="test")
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=gnot_collate_fn)
    batch = next(iter(loader))
    batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(batch)

    pred_field = outputs['field']   # [B, N, K]
    pred_freq  = outputs['freq']    # [B, K]  — sorted ascending
    true_field = batch['Y_field']   # [B, N, K]
    true_freq  = batch['Y_freq']    # [B, K]
    mask       = batch['Mask']      # [B, N]

    B, N, K = pred_field.shape

    freq_errs, field_errs = [], []
    deg_pairs = []

    for b in range(B):
        # Hungarian eşleme (frekans bazlı)
        cost = torch.abs(pred_freq[b].unsqueeze(1) - true_freq[b].unsqueeze(0))  # [K,K]
        row, col = linear_sum_assignment(cost.cpu().numpy())
        perm = col  # pred[perm[i]] ↔ true[i]

        f_p = pred_freq[b, perm]
        f_t = true_freq[b]
        freq_errs.append((f_p - f_t).abs().mean().item())

        m = mask[b].float()
        for i in range(K):
            e_p = pred_field[b, :, perm[i]] * m
            e_t = true_field[b, :, i] * m
            rel = (e_p - e_t).norm() / (e_t.norm().clamp(min=1e-8))
            # sign ambiguity
            rel2 = (e_p + e_t).norm() / (e_t.norm().clamp(min=1e-8))
            field_errs.append(min(rel.item(), rel2.item()))

        # Near-degenerate pair check
        for i in range(K):
            for j in range(i+1, K):
                gap = abs(true_freq[b, i].item() - true_freq[b, j].item())
                mean_f = true_freq[b].mean().abs().item() + 1e-8
                if gap / mean_f < 0.05:
                    # Grassmannian measure: 1 − |cos²|  for 2D
                    u = pred_field[b, :, perm[i]] * m
                    v = pred_field[b, :, perm[j]] * m
                    t1 = true_field[b, :, i] * m
                    t2 = true_field[b, :, j] * m
                    # sin² of principal angle  (1 = orthogonal complement, 0 = perfect)
                    M = torch.stack([u,v], 1).T @ torch.stack([t1,t2], 1)  # [2,2]
                    sv = torch.linalg.svdvals(M / (M.norm() + 1e-8))
                    gr_dist = (2 - (sv**2).sum()).item()
                    deg_pairs.append(gr_dist)

    print(f"{'='*55}")
    print(f"  FREKANS HATASI (norm)  : {np.mean(freq_errs):.4f} ± {np.std(freq_errs):.4f}")
    print(f"  ALAN REL-L2            : {np.mean(field_errs):.4f} ± {np.std(field_errs):.4f}")
    if deg_pairs:
        print(f"  NEAR-DEG GRASSMANNIAN  : {np.mean(deg_pairs):.4f}  (0=mükemmel, 2=rastgele)")
    else:
        print(f"  NEAR-DEG GRASSMANNIAN  : N/A  (bu batchte near-degenerate çift yok)")
    print(f"{'='*55}")
else:
    print("Checkpoint bulunamadı, önce eğitimi çalıştır.")


## 8. Checkpoint'u Google Drive'a Kaydet

In [ ]:
# Eğitilmiş modeli Drive'a kaydet
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE and best_ckpt:
    drive_save_dir = "/content/drive/MyDrive/rf_cavity_checkpoints"
    os.makedirs(drive_save_dir, exist_ok=True)
    
    save_name = os.path.basename(best_ckpt)
    !cp "{best_ckpt}" "{drive_save_dir}/{save_name}"
    
    # Inference sonuçlarını da kaydet
    !cp -r inference_results "{drive_save_dir}/inference_results"
    
    print(f"✅ Checkpoint kaydedildi: {drive_save_dir}/{save_name}")
    print(f"✅ Inference sonuçları kaydedildi: {drive_save_dir}/inference_results/")